In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_excel("../data/Telco-Customer-Churn.xlsx")

df.head()

### สร้าง Target

In [ ]:
y = df["Churn Label"]

### ตัด Churn-related columns ออก

In [ ]:
X = df.drop(columns=[
    "Churn Label",
    "Churn Value",
    "Churn Score",
    "Churn Reason"
])

### ตรวจ Target

In [ ]:
df["Churn Label"].value_counts()

### แยก X และ y

เหตุผลคือไม่ต้องการให้ข้อมูลที่เกี่ยวข้องกับผลลัพธ์โดยตรงรั่วเข้าไปใน Model

In [ ]:
y = df["Churn Label"]

X = df.drop(columns=[
    "Churn Label",
    "Churn Value",
    "Churn Score",
    "Churn Reason"
])

#### ดูว่า X เหลืออะไรบ้าง และตัดบ้าง feature ที่ไม่จำเป็นต่อการเทรนโมเดล

In [ ]:
# Convert Total Charges to numeric
df["Total Charges"] = pd.to_numeric(
    df["Total Charges"],
    errors="coerce"
)

# Fill missing Total Charges
df["Total Charges"] = df["Total Charges"].fillna(0)

# Target
y = df["Churn Label"]

# Features
X = df.drop(columns=[
    "Churn Label",
    "Churn Value",
    "Churn Score",
    "Churn Reason",
    "CustomerID",
    "Count",
    "Country",
    "State",
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude",
    "CLTV"
])

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X.info()

In [ ]:
X.head()

## Train/Test Split + Encoding

### 1. Import

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

### 2. แยก Numerical / Categorical

In [ ]:
numeric_features = [
    "Tenure Months",
    "Monthly Charges",
    "Total Charges"
]

categorical_features = [
    col for col in X.columns
    if col not in numeric_features
]

In [ ]:
print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

### 3. แปลง Target เป็น 0/1

In [ ]:
y = y.map({
    "No": 0,
    "Yes": 1
})

In [ ]:
y.value_counts()

### 4. แบ่ง Train / Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

### 5. สร้าง Preprocessing Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

Contract

Month-to-month
One year
Two year

Contract_Month-to-month
Contract_One year
Contract_Two year

### 6. สร้าง Model แรก

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

### 7. Train

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

### 8. Results

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Baseline Model: Logistic Regression

The Logistic Regression baseline achieved an accuracy of 80.2%.

For the churn class, the model achieved:
- Precision: 64%
- Recall: 57%
- F1-score: 60%

Although the overall accuracy is relatively high, the recall for
churned customers is only 57%. This means the model misses a
considerable number of customers who actually churn.

Since identifying potential churn customers is an important business
objective, improving recall for the churn class will be a key focus
when comparing other models.

### 9. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

print(cm)

In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Churn", "Churn"]
)

disp.plot()
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

917 → ลูกค้าที่ไม่ Churn และ Model ทายถูกว่าไม่ Churn

118 → ลูกค้าที่ไม่ Churn แต่ Model ทายว่าจะ Churn → False Positive

161 → ลูกค้าที่ Churn จริง แต่ Model ทายว่าไม่ Churn → False Negative 

213 → ลูกค้าที่ Churn และ Model ทายถูก → True Positive

### Confusion Matrix Analysis

The Logistic Regression model correctly identified 213 churned
customers, but missed 161 customers who actually churned.

The relatively high number of false negatives indicates that the
baseline model still misses a considerable number of potential
churn customers.

Therefore, improving churn recall will be an important objective
when evaluating more advanced models.

## ROC-AUC

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", roc_auc)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))

plt.plot(
    fpr,
    tpr,
    label=f"Logistic Regression (AUC = {roc_auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend()
plt.tight_layout()
plt.show()

### ROC-AUC Evaluation

The Logistic Regression model achieved a ROC-AUC score of 0.849,
indicating good overall discrimination between churned and
non-churned customers.

However, the recall for the churn class is 57%, meaning that the
model still misses a considerable number of customers who actually
churn.

Therefore, additional models will be evaluated to determine whether
we can improve churn detection while maintaining good overall
performance.

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

### สร้าง Model:

In [ ]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

### Train

In [ ]:
rf_model.fit(X_train, y_train)

### Predict:

In [ ]:
rf_pred = rf_model.predict(X_test)

### ดูผล:

In [ ]:
print("Accuracy:", accuracy_score(y_test, rf_pred))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

### ROC-AUC:

In [ ]:
rf_prob = rf_model.predict_proba(X_test)[:, 1]

rf_auc = roc_auc_score(y_test, rf_prob)

print("Random Forest ROC-AUC:", rf_auc)

## Model Comparison

The Logistic Regression model achieved higher overall accuracy
(80.2%) and ROC-AUC (0.849) than the Random Forest model
(77.5% accuracy and 0.834 ROC-AUC).

However, Random Forest achieved a higher recall for the churn class,
increasing from 57% to 65%, and a slightly higher F1-score from 0.60
to 0.61.

Since identifying potential churn customers is an important business
objective, Random Forest shows an advantage in churn detection.
However, additional models should be evaluated before selecting
the final model.

# XGBoost

### Recap
Logistic Regression
       │
       ├── Accuracy 80.2%
       ├── Recall 57%
       └── AUC 0.849
       
Random Forest
       │
       ├── Accuracy 77.5%
       ├── Recall 65% ⭐
       └── AUC 0.834
       
XGBoost
       │
       └── ??? ← 

In [ ]:
from xgboost import XGBClassifier

### 1. XGBoost Model

In [ ]:
xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ))
    ]
)

n_estimators=300 → จำนวนต้นไม้

learning_rate=0.05 → เรียนรู้ทีละน้อย

max_depth=4 → จำกัดความลึกของต้นไม้

subsample=0.8 → ใช้ข้อมูล 80% ต่อรอบ

colsample_bytree=0.8 → ใช้ features 80% ต่อ tree

### 2. Train

In [ ]:
xgb_model.fit(X_train, y_train)

### 3. Predict

In [ ]:
xgb_pred = xgb_model.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, xgb_pred))

print("\nClassification Report:")
print(classification_report(y_test, xgb_pred))

### 4. ROC-AUC

In [ ]:
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

xgb_auc = roc_auc_score(y_test, xgb_prob)

print("XGBoost ROC-AUC:", xgb_auc)

## Model Comparison

Three classification models were evaluated: Logistic Regression,
Random Forest, and XGBoost.

| Model | Accuracy | Recall (Churn) | F1-score (Churn) | ROC-AUC |
|---|---:|---:|---:|---:|
| Logistic Regression | 80.2% | 57% | 0.60 | 0.849 |
| Random Forest | 77.5% | 65% | 0.61 | 0.834 |
| XGBoost | 80.5% | 56% | 0.60 | 0.856 |

Random Forest achieved the highest recall for the churn class,
while XGBoost achieved the highest ROC-AUC and overall accuracy.

Because the primary business objective is to identify customers
at risk of churn, recall for the churn class is particularly
important. Therefore, Random Forest is selected as the initial
candidate for the final model.

## Feature Importance

In [ ]:
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()

importances = rf_model.named_steps["classifier"].feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

feature_importance.head(15)

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    feature_importance.head(15)["Feature"][::-1],
    feature_importance.head(15)["Importance"][::-1]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")

plt.tight_layout()
plt.show()

## Feature Importance Analysis

The Random Forest model identified Total Charges, Tenure Months,
and Monthly Charges as the three most important features.

Contract type, online security, tech support, and payment method
were also important predictors.

These findings are consistent with the earlier exploratory analysis,
where shorter tenure, higher monthly charges, month-to-month contracts,
electronic check payments, and lack of additional services were
associated with higher churn rates.

Feature importance indicates which features contributed to the model's
predictions, but does not imply that these features directly cause
customer churn.

## Threshold Tuning ของ Random Forest

### 1. Probability ของ Random Forest

In [ ]:
rf_prob = rf_model.predict_proba(X_test)[:, 1]

### 2. ทดลองหลาย Threshold

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

results = []

for threshold in thresholds:
    predictions = (rf_prob >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions)
    })

threshold_results = pd.DataFrame(results)

threshold_results

### 3. ทำกราฟ    

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    threshold_results["Threshold"],
    threshold_results["Precision"],
    marker="o",
    label="Precision"
)

plt.plot(
    threshold_results["Threshold"],
    threshold_results["Recall"],
    marker="o",
    label="Recall"
)

plt.plot(
    threshold_results["Threshold"],
    threshold_results["F1"],
    marker="o",
    label="F1-score"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Threshold Tuning - Random Forest")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Threshold Tuning

The default classification threshold of 0.50 resulted in a churn
recall of 65.5%.

Several thresholds were evaluated to improve the model's ability to
identify customers at risk of churn.

A threshold of 0.35 achieved the highest F1-score (0.620) among the
tested thresholds, with a precision of 50.1% and recall of 81.3%.

Compared with the default threshold of 0.50, lowering the threshold
to 0.35 substantially improves churn recall from 65.5% to 81.3%,
although precision decreases from 56.8% to 50.1%.

Because identifying potential churn customers is the primary business
objective, a threshold of 0.35 was selected for the final model.

## Final Prediction

In [ ]:
FINAL_THRESHOLD = 0.35

In [ ]:
final_pred = (rf_prob >= FINAL_THRESHOLD).astype(int)

In [ ]:
print(classification_report(y_test, final_pred))

In [ ]:
final_cm = confusion_matrix(y_test, final_pred)

print(final_cm)

## Final Model Evaluation

After threshold tuning, the classification threshold was reduced
from 0.50 to 0.35 to prioritize the detection of churn customers.

At the 0.35 threshold, the model achieved a churn recall of 81.3%,
compared with 65.5% at the default 0.50 threshold.

The number of false negatives decreased from 161 to 70, meaning
the model was able to identify substantially more customers who
actually churned.

However, false positives increased from 118 to 303, resulting in
lower overall accuracy and precision.

This represents a precision-recall trade-off. The 0.35 threshold
was selected because the primary business objective is to identify
as many potential churn customers as possible.